#  Supercomputing Internet LLM Fine-Tuning LoRA Example

This notebook is an English, GitHub-friendly translation of the original document. The Python code cells are kept unchanged as requested.


## Overview

This notebook fine-tunes the model `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` (1.5B parameters) with **LoRA** on a sentiment-analysis dataset.

Key points of this tutorial:

- The base model is downloaded **through the Hugging Face mirror** (`hf-mirror.com`) and saved to `D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B`.
- Fine-tuning uses 4-bit quantization (NF4) + LoRA so the whole run fits in ~6 GB of GPU memory.
- Everything runs **locally** with the standard Hugging Face `Trainer` workflow (no cloud platform required).
- The dataset `twitter-airline-sentimentSentiment_Analysis.csv` is already downloaded next to this notebook.

This is public-interest code, generated with the help of DeepSeek and tested in an example environment.

## 1. Market context for large language models

Outside of API usage from major frontier vendors, most civilian LLM research and applications are already clearly differentiated at the 500B scale and below (excluding TAALAS-related techniques).

1. **Hundreds-of-billions-scale models (100B+ to 500B)** represented by OpenAI GPT-OSS-120B. This track currently emphasizes low bit precision, such as 4-bit native training precision, strong information encoding, and improved information efficiency without sacrificing quality as the baseline. For example, GPT-OSS-120B can be used as a benchmark and compared directly with GPT-4-class models. Without strong model optimization or theoretical support, domestic models in China may not challenge trillion-parameter models; even if they do, they may still find it difficult to compete with GPT-OSS-120B. This reflects the current mathematical and practical limitations of large-model development in China, not a denial of 10T- or 100T-parameter models. However, hundreds-of-billions-scale models usually require multi-GPU operation, which creates a technical barrier.

2. **Tens-of-billions-scale models** represented by OpenAI GPT-OSS-20B. This group includes the classic Microsoft Phi-4 series at 14B and 7B, as well as distilled models from the Qwen and DeepSeek families, which are mainstream offerings from first- and second-tier vendors. With industrial 4-bit support, these models can run directly on consumer-grade computers (30–50 GB). Their token generation speed is comparable to a single user's information-processing speed. In particular, their fine-tuning compute requirements are relatively small, making them suitable for small and medium-sized enterprises to perform secondary development directly at the model layer. In some cases, the cost can be as low as a few thousand RMB. After efficiency improvements or sub-4-bit optimization, they may become mainstream for edge computing.

3. **Billion-scale models** represented by DeepSeek. A classic example is the DeepSeek 1.5B distilled model, which usually occupies 3 GB of memory or less. After further optimization, these models can be deployed directly on mobile phones or small computers. Their technical parameters are comparable to tens-of-billions-scale models, making them suitable for teaching purposes with extremely low training cost, typically within tens of RMB. The techniques learned here can be quickly transferred to tens-of-billions-scale model workflows, making this an effective way to experiment and iterate.


## 2. Local setup steps

The following steps set up a **local** fine-tuning environment (Windows + Python 3.12, e.g. the `agentic_ai` kernel in VS Code).

1. Install the required libraries (already installed in this environment):

   ```
   pip install torch transformers accelerate peft bitsandbytes datasets trl scikit-learn pandas
   ```

   If you are in China, you can speed up pip with the Aliyun mirror:

   ```
   pip config set global.index-url https://mirrors.aliyun.com/pypi/simple/
   ```

2. **Download the base model through the Hugging Face mirror** (`hf-mirror.com`) into `D:\HuggingfaceDownload\...`:

   The first code cell (Section 3) sets `HF_ENDPOINT=https://hf-mirror.com` and `HF_HOME=D:\HuggingfaceDownload\.cache\huggingface` **before** importing `transformers`, so every model download goes through the mirror and the cache lands on the `D:` drive. The unpacked model is then saved to `D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B`.

3. **Dataset**: the CSV file `twitter-airline-sentimentSentiment_Analysis.csv` has already been downloaded to

   ```
   C:\Deepin\Programming\20260803 AgenticAILLMVisionModel2026Tutorials\tutorials\01-llm-transformer-training\twitter-airline-sentimentSentiment_Analysis.csv
   ```

   which is the same folder as this notebook, so the code reads it with a relative path (with an absolute-path fallback).

4. Run the notebook cells in order:

   - Sections 3–5: download and test the base model.
   - Sections 6–7: load the dataset and prepare the fine-tuning data.
   - Sections 8–13: 4-bit quantization, LoRA setup, and training.
   - Sections 14–15: test the fine-tuned model.

## 3. Download the base model

Download the model `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` **through the Hugging Face mirror** (`HF_ENDPOINT=https://hf-mirror.com`) and save it locally to:

`D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B`

The download cache also lives on the `D:` drive (`D:\HuggingfaceDownload\.cache\huggingface`).

In [ ]:
# =============================================================================
# SECTION 3: Download the base model from Hugging Face mirror
# =============================================================================
# This cell downloads DeepSeek-R1-Distill-Qwen-1.5B (~3 GB) via the
# hf-mirror.com mirror (accessible in China) and saves it locally so
# subsequent runs can load it without any network access.

import os

# ---- Use the Hugging Face mirror (hf-mirror.com) for model downloads ----
# IMPORTANT: must be set BEFORE importing transformers / huggingface_hub
# HF_ENDPOINT: redirects all HF downloads to the China-accessible mirror
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
# HF_HOME: stores cached model blobs on the D: drive to save C: drive space
os.environ["HF_HOME"] = r"D:\HuggingfaceDownload\.cache\huggingface"



# Import core Hugging Face classes — AutoModelForCausalLM loads any
# GPT-style (decoder-only) language model; AutoTokenizer loads its tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Define local save directory for the 1.5B model
# After download, this folder will contain config.json, model.safetensors,
# tokenizer.json, and other necessary files
local_model_dir = r"D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B"

# ✅ Correct Model Name (1.5 Billion parameters)
# DeepSeek-R1-Distill-Qwen-1.5B is a distilled, smaller version of the
# DeepSeek-R1 reasoning model, based on the Qwen2 architecture
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

print(f"Downloading via HF mirror: {os.environ['HF_ENDPOINT']}")
print(f"Model: {model_name} (Official size: 1.54B parameters)")

# Check whether the model was already downloaded — skip if it exists locally
if os.path.isdir(local_model_dir) and os.listdir(local_model_dir):
    print(f"Local model already exists at {local_model_dir}, skipping download...")
else:
    # Download the tokenizer first — it's small (~a few MB)
    # trust_remote_code=True is needed because DeepSeek models use custom code
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    # Download the full model: bfloat16 halves VRAM usage vs float32
    # device_map="auto" lets Hugging Face pick GPU first, then CPU
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,   # bfloat16 is safe and memory efficient
        device_map="auto"             # automatic device placement
    )

    # Persist both tokenizer and model to disk for future offline use
    tokenizer.save_pretrained(local_model_dir)
    model.save_pretrained(local_model_dir)
    print(f"Model saved to {local_model_dir}")

## 4. Hugging Face mirror notes

Because the model is downloaded through the mirror (`hf-mirror.com`), a few points are worth knowing:

### 4.1 Environment variables

| Variable | Value in this notebook | Effect |
| --- | --- | --- |
| `HF_ENDPOINT` | `https://hf-mirror.com` | All downloads use the mirror instead of `huggingface.co` |
| `HF_HOME` | `D:\HuggingfaceDownload\.cache\huggingface` | Cache and local config live on the `D:` drive |

Both are set with `os.environ[...]` in the first code cell, **before** `transformers` is imported.

### 4.2 Where files are stored

- The mirror cache (blobs): `D:\HuggingfaceDownload\.cache\huggingface\hub`
- The unpacked model used by the rest of this notebook: `D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B`

### 4.3 Work fully offline afterwards

Once the model is saved locally, the notebook loads it from the local directory, so no network is needed. To make `huggingface_hub` fail loudly instead of trying the network:

```powershell
# in the terminal (PowerShell)
$env:HF_HUB_OFFLINE = "1"
```

### 4.4 Find large files in the download directory

```powershell
Get-ChildItem -Recurse D:\HuggingfaceDownload | Sort-Object Length -Descending | Select-Object -First 20 FullName, Length
```

This shows the 20 largest files and folders under `D:\HuggingfaceDownload`.

## 5. Test the downloaded model

Before fine-tuning, we verify that the base model loads correctly from the local directory and can generate coherent text. This is a **sanity check** — we ask the model a general-knowledge question ("Explain quantum computing in simple terms") and inspect the output. If this works, we know the model files are intact and the environment is properly configured.

The cell below:
1. Loads the tokenizer and model from the local `D:\HuggingfaceDownload\...` directory (no network needed)
2. Tokenizes a prompt string into token IDs
3. Runs autoregressive generation with `model.generate()`
4. Decodes the output tokens back into readable text

In [ ]:
# =============================================================================
# SECTION 5: Sanity-check the downloaded model with a simple text generation
# =============================================================================
# This cell loads the locally saved model and runs a single forward generation
# to confirm everything works before we proceed to fine-tuning.

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the model from the local directory (no network needed)
local_model_dir = r"D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B"

# Re-load tokenizer and model — this time from local disk, not from HF hub
tokenizer = AutoTokenizer.from_pretrained(local_model_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    local_model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Example prompt — a general-knowledge question to test the base model
input_text = "Explain quantum computing in simple terms"
# Tokenize: convert string → tensor of token IDs, move to same device as model
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

# Generate text using the base (not yet fine-tuned) model
# max_new_tokens=200: generate up to 200 new tokens after the prompt
# temperature=0.7: moderate randomness (0=deterministic, 1=creative)
# do_sample=True: use sampling instead of greedy decoding
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id  # use EOS token for padding
)

# Decode the generated token IDs back into human-readable text
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 6. Load the dataset

We use the **Twitter Airline Sentiment** dataset — ~14,000 tweets about US airlines, each labeled as *positive*, *negative*, or *neutral*. For faster fine-tuning, we only use the first **5,000 rows**.

The dataset columns are:
- `tweet_id` — unique identifier
- `sentiment` — the label we want the model to predict (positive/negative/neutral)
- `author` — username of the tweet author
- `content` — the tweet text (our model input)

The cell below reads the CSV with pandas and prints a preview of the first few rows.

In [ ]:
# =============================================================================
# SECTION 6: Load the Twitter Airline Sentiment dataset
# =============================================================================
# This dataset contains ~14,000 tweets about US airlines, each labeled with
# one of three sentiments: positive, negative, or neutral.
# We only use the first 5,000 rows for faster fine-tuning.

import os
import pandas as pd

# The CSV has already been downloaded next to this notebook.
# Try the relative path first, then fall back to the absolute path.
csv_path = "twitter-airline-sentimentSentiment_Analysis.csv"
if not os.path.exists(csv_path):
    csv_path = r"C:\Deepin\Programming\20260803 AgenticAILLMVisionModel2026Tutorials\tutorials\01-llm-transformer-training\twitter-airline-sentimentSentiment_Analysis.csv"

# Read the CSV into a pandas DataFrame
df = pd.read_csv(csv_path)
# Use only the first 5,000 rows for faster fine-tuning
first_50 = df.head(5000)
print(f"Loaded {len(first_50)} rows from {csv_path}")
# Preview the first few rows: tweet_id, sentiment label, author, and content
print(first_50[['tweet_id', 'sentiment', 'author', 'content']].head())

The CSV file already exists in the same folder as this notebook:

`C:\Deepin\Programming\20260803 AgenticAILLMVisionModel2026Tutorials\tutorials\01-llm-transformer-training\twitter-airline-sentimentSentiment_Analysis.csv`

## 7. Prepare the fine-tuning data — Prompt-Format Alignment

This is where the **alignment** happens. The key principle: the model must see the **exact same prompt structure** at training time and inference time. Any mismatch will degrade performance.

### The Instruction Template (Training Format)

We wrap every tweet+label pair in a fixed instruction template:

```
Instruction: Analyze the sentiment of the following tweet:
Input: <tweet text>
Output: <sentiment label>
```

This three-part structure teaches the model a specific pattern:
1. **Instruction** — tells the model *what task* to perform
2. **Input** — provides the *data* to analyze (the tweet content)
3. **Output** — shows the *expected answer* (the sentiment label)

### Why Format Alignment Matters

During inference (Section 14), the prompt ends at `Output:` — the model then completes the sequence:

```
Instruction: Analyze the sentiment of the following tweet:
Input: @VirginAmerica I <3 you!
Output:                              ← model generates from here
```

The model has learned that after seeing `Instruction:` + `Input:` + `Output:`, the next token should be a sentiment label. If you change the template, add extra text, or miss the newlines, the model will produce unexpected output because the pattern it memorized no longer matches.

This is the fundamental alignment constraint in instruction tuning: **training prompt format ≡ inference prompt format**.

### Why the Old Code Hallucinated Extra Text (and how Loss Masking fixes it)

Each training example is a **separate padded sequence** — they are not concatenated. The EOS token was always present. The real problem: the old code (`labels = input_ids.clone()`) trained the model to predict **every token** in the full template:

```
Example (padded to 512 tokens):
Instruction: Analyze... Input: tweet Output: worry <EOS> [PAD] [PAD]...
└─────── ALL of these contributed to loss (except PADs) ───────┘
```

The model learned to predict not just `worry`, but also `Instruction:`, `Analyze`, `the`, `sentiment`, etc. Its **entire statistical distribution** was shaped by the template pattern. At inference time, `max_new_tokens=10` forces it to generate 10 tokens. After correctly outputting `worry`, the remaining budget gets filled with the next-most-likely text it was trained on: **more template boilerplate**.

```
Predicted: worry\n\nInput: I'm going to miss my...
                └───── model's template-shaped distribution ─────┘
```

**The fix in Section 9:** we mask all template tokens with `-100` (ignored by the loss), so the model's distribution is ONLY shaped by the sentiment label itself. No more template hallucination — the remaining token budget just produces whitespace or stops.

The cell below iterates over the 5,000 rows, builds these formatted strings with the fixed template, and wraps them in a Hugging Face `Dataset` object.

In [ ]:
# =============================================================================
# SECTION 7: Prompt-format alignment — train the model on a fixed template
# =============================================================================
# CRITICAL CONCEPT: The prompt format used here MUST match the format used
# at inference time (in Section 14). This is the alignment constraint:
# the model learns to map "Instruction:\nInput:\nOutput:" → sentiment label.
# If the inference prompt differs even slightly (different wording, missing
# newlines, extra spaces), the model may produce garbage.
#
# NOTE ON EOS VS LOSS MASKING:
# Each example is a SEPARATE padded sequence — they are NOT concatenated.
# The EOS token was always present; that's not the issue. The bug happened
# because the old code (labels = input_ids.clone()) trained the model to
# predict EVERY token — "Instruction:", "Analyze", "the", etc. The model's
# entire statistical distribution learned the template pattern. At inference,
# max_new_tokens=10 forces it to generate 10 tokens; after correctly saying
# "worry", the remaining budget gets filled with the next-most-likely text
# it was trained on: more template. Loss masking (Section 9) fixes this by
# setting template token labels to -100 so the model ONLY learns the label.

# The instruction tells the model what task to perform
instruction = "Analyze the sentiment of the following tweet:"

# Each training example uses the EXACT 3-part format:
#   Instruction: <task description>
#   Input: <tweet content>
#   Output: <sentiment label>
# The model learns that "Output:" is followed by the answer.
formatted_texts = []
for idx, row in first_50.iterrows():
    text = f"Instruction: {instruction}\nInput: {row['content']}\nOutput: {row['sentiment']}"
    formatted_texts.append(text)

# Convert the list of strings into a Hugging Face Dataset object
# This is the native data format expected by the Trainer API
from datasets import Dataset
dataset = Dataset.from_dict({"text": formatted_texts})

# Print the first example to verify the format
print(dataset[0]['text'])

## 8. 4-bit quantization and LoRA model setup

This is the core of the memory-efficient fine-tuning strategy. Two techniques work together:

### 8.1 4-bit NF4 Quantization (BitsAndBytes)
The base model's 1.5 billion float16 parameters (~3 GB) are compressed to 4-bit integers (~0.75 GB). During computation, weights are temporarily dequantized back to bfloat16. The NF4 (NormalFloat4) data type is optimized for normally-distributed weights and outperforms plain int4.

### 8.2 LoRA (Low-Rank Adaptation)
Instead of updating all 1.5B parameters, LoRA inserts small trainable matrices (`A` and `B`) into the attention layers. Only these matrices are updated during training — the original weights stay **frozen**. The key parameters:
- **r (rank)** = 8: The low-rank dimension of the adapter matrices
- **lora_alpha** = 32: Scaling factor (effective learning rate ∝ alpha/r = 4)
- **target_modules**: We apply LoRA to `q_proj` and `v_proj` (query and value projections in self-attention) — these are the most impactful layers for adaptation

The result: only ~0.1% of parameters are trainable, making fine-tuning possible on a single consumer GPU with ~6 GB VRAM.

In [ ]:
# =============================================================================
# SECTION 8: 4-bit quantization + LoRA setup
# =============================================================================
# Two key techniques make fine-tuning a 1.5B model possible on ~6 GB VRAM:
# 1. 4-bit NF4 quantization — compresses the base model weights from 16-bit
#    to 4-bit, reducing memory ~4×.
# 2. LoRA (Low-Rank Adaptation) — instead of updating all 1.5B parameters,
#    we only train small "adapter" matrices (~0.1% of the total parameters).
#    The base weights stay frozen.

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig          # ← configures 4-bit / 8-bit quantization
)
from peft import LoraConfig, get_peft_model   # ← PEFT = Parameter-Efficient Fine-Tuning

# Check if a CUDA-capable GPU is available
use_cuda = torch.cuda.is_available()
print(f"CUDA available: {use_cuda}")

# ---------------------------------------------------------------------------
# 4-bit quantization config — only works on GPU (CUDA)
# ---------------------------------------------------------------------------
# - load_in_4bit=True: weights are stored as 4-bit integers
# - bnb_4bit_quant_type="nf4": "NormalFloat4" — better than plain int4
# - bnb_4bit_compute_dtype=torch.bfloat16: dequantize to bfloat16 during compute
# - bnb_4bit_use_double_quant=True: quantize the quantization constants too
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load model with quantization (local copy downloaded via HF mirror)
model_name = r"D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# Set pad_token to eos_token — causal LMs often don't have a dedicated pad token
tokenizer.pad_token = tokenizer.eos_token

if use_cuda:
    # GPU path: load with 4-bit quantization for maximum memory savings
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
else:
    # CPU fallback: no 4-bit quantization (BitsAndBytes is CUDA-only)
    # Load in bfloat16 instead to save some memory vs float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

# ---------------------------------------------------------------------------
# LoRA (Low-Rank Adaptation) configuration
# ---------------------------------------------------------------------------
# - r=8: rank of the low-rank decomposition matrices (higher = more capacity)
# - lora_alpha=32: scaling factor for the LoRA update (alpha/r = effective lr scale)
# - target_modules=["q_proj", "v_proj"]: apply LoRA to query & value projections
#   in the attention layers — these are the most impactful for adaptation
# - lora_dropout=0.05: dropout on LoRA layers for regularization
# - bias="none": don't train bias terms
# - task_type="CAUSAL_LM": we're fine-tuning a causal (autoregressive) language model
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],   # typical for DeepSeek/Qwen architectures
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap the quantized base model with LoRA adapters
# After this call, only the LoRA parameters will be trained — base weights frozen
model = get_peft_model(model, lora_config)
# Print trainable vs total parameters — expect ~0.1% trainable
model.print_trainable_parameters()  # should show ~0.1% trainable

## 9. Prepare the training data — with loss masking

We now **tokenize** the formatted text strings into numerical token IDs that the model can process. Each example is padded independently to 512 tokens.

The key steps are:

1. **Tokenization**: Each text string is split into tokens using the model's vocabulary, then truncated or padded to exactly 512 tokens. The tokenizer automatically appends `<|endoftext|>` (EOS).

2. **Label creation with loss masking** (THE CRITICAL FIX):

   **What the old code did wrong**: `labels = input_ids.clone()` computed loss on every token — `Instruction:`, `Analyze`, `the`, `sentiment`, etc. The model learned to predict the entire template pattern, not just the sentiment label.

   **Why that caused hallucination**: At inference, `max_new_tokens=10` forces 10 tokens of generation. After correctly predicting `worry`, the model fills its remaining budget with whatever text is statistically most likely from training — which is more template text like `\n\nInput: I'm going to miss my`.

   **How loss masking fixes it**:
   - All tokens **before** `\nOutput:` are set to `-100` (Hugging Face ignores `-100` in the cross-entropy loss)
   - Only the sentiment label tokens contribute to the loss
   - The model's distribution is now ONLY shaped by the labels, not the template boilerplate

   ```
   "Instruction: ... Output: positive<EOS>[PAD]..."
   labels:   [-100] ...    [-100]  [positive] [<EOS>]  [-100] ...
              └─ ignored ──────┘  └── trained ──┘  └─ ignored ─┘
   ```

3. **Format conversion**: The dataset is set to PyTorch format so tensors are created automatically when batching.

The cell below defines a `tokenize_function` and a `set_labels` function that applies the loss mask by locating the `\nOutput:` marker in each tokenized sequence.

In [ ]:
# =============================================================================
# SECTION 9: Tokenize the dataset and create labels for causal LM training
# =============================================================================
# For causal language modeling, the "labels" are simply the input_ids shifted
# by one position. The model predicts token N+1 given tokens 1..N.
#
# KEY FIX — Loss Masking:
#
# Each example is a SEPARATE padded sequence (NOT concatenated with others).
# The EOS token was always automatically appended by the tokenizer — not the issue.
#
# PROBLEM (old code: labels = input_ids.clone()):
#   Every token contributed to loss: "Instruction:", "Analyze", "the", etc.
#   The model's ENTIRE statistical distribution learned the template pattern.
#   At inference, max_new_tokens=10 forces 10 tokens of generation. After
#   correctly saying "worry", the remaining budget gets filled with whatever
#   text is statistically most likely from training — which is more template
#   boilerplate (e.g. "\n\nInput: I'm going to miss my").
#
# FIX (loss masking = set pre-Output tokens to -100):
#   The model's distribution is ONLY shaped by the sentiment label portion.
#   Extra generation budget just produces <EOS>/whitespace, not template text.

# Encode the output-marker string to find its token IDs
# We'll look for "\nOutput:" in each tokenized sequence to know where to mask
output_marker = "\nOutput:"
output_marker_ids = tokenizer.encode(output_marker, add_special_tokens=False)

def tokenize_function(examples):
    """Convert raw text strings into token IDs with fixed-length padding."""
    tokenized = tokenizer(
        examples["text"],
        truncation=True,          # truncate texts longer than max_length
        padding="max_length",     # pad all sequences to exactly max_length
        max_length=512,           # 512 tokens is a standard context window
        return_tensors=None       # return Python lists; data collator handles tensor conversion
    )
    return tokenized

# Apply tokenization to the entire dataset in batched mode for speed
# remove_columns=["text"] drops the raw text column — we only need token IDs
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
# Set the PyTorch format so the Trainer can directly use these columns
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask"])

def set_labels(example):
    """Copy input_ids to labels for causal LM training, but MASK everything
    before '\nOutput:' so the model only learns to predict the sentiment label.
    The model internally shifts labels by 1 position for next-token prediction.

    Masking strategy:
    - tokens BEFORE '\nOutput:' → label = -100 (ignored in loss)
    - tokens FROM '\nOutput:' onward (including the sentiment label) → label = token ID
    This ensures the model only learns: given the prompt, what comes after 'Output:'?

    Example (conceptual):
      "Instruction: ... Output: positive<EOS>[PAD]..."
      labels: [-100, -100, ..., -100, positive, <EOS>, -100, ...]
                └─ model trains on NONE of this ─┘└── model trains on this ─┘
    """
    input_ids = example["input_ids"]

    # Find where "\nOutput:" starts in the tokenized sequence
    # We search for the marker token IDs as a subsequence
    marker_len = len(output_marker_ids)
    output_start = -1
    for i in range(len(input_ids) - marker_len + 1):
        if input_ids[i:i+marker_len].tolist() == output_marker_ids:
            output_start = i
            break

    if output_start != -1:
        # Mask everything before (and including) the "\nOutput:" marker
        # We include the marker itself so the loss isn't computed on "Output:"
        labels = [-100] * (output_start + marker_len)
        # Keep the real labels for the sentiment portion (after "\nOutput:")
        labels.extend(input_ids[output_start + marker_len:].tolist())
        # Truncate/pad to match the fixed max_length (512)
        labels = labels[:512]
    else:
        # Fallback: if marker not found (shouldn't happen), mask everything
        labels = [-100] * len(input_ids)

    example["labels"] = labels
    return example

# Add the "labels" column to every example with proper loss masking
tokenized_dataset = tokenized_dataset.map(set_labels)

### 9.1 Understanding Loss Masking — Complete Discussion

This section explains **why** loss masking works, **how** it interacts with backpropagation, and **why it doesn't break alignment**. These are the most frequently misunderstood concepts in instruction fine-tuning.

---

#### What is Loss Masking?

**Loss masking** means setting certain token labels to $-100$ in Hugging Face. The cross-entropy loss function **ignores** positions where `labels == -100` — those positions contribute $0$ to the loss and receive $0$ gradient.

In our code:

```python
labels = [-100] * (output_start + marker_len)  # mask Instruction, Input, Output:
labels.extend(input_ids[output_start + marker_len:])  # keep sentiment label + EOS
```

The masked tokens still flow through the forward pass (self-attention sees them), but the loss is only computed on the sentiment label.

---

#### Was the Model Hallucinating?

**Not in the classical sense** (making up false facts). The model was repeating **statistically learned template patterns**. Because the old code computed loss on *every* token, the model's probability distribution was shaped by the full template: `Instruction:`, `Analyze`, `the`, `Input:`, etc.

At inference, `max_new_tokens=10` forced 10 tokens of generation. After correctly predicting `worry`, the remaining budget was filled with the next-most-likely text the model was trained on — more template boilerplate like `\n\nInput: I'm going to miss my`.

The model wasn't "making things up." It was doing exactly what it was trained to do: predict the most probable continuation given the context.

---

#### Forward Pass vs. Backward Pass — The Key Distinction

This is the single most important concept to understand:

```
┌─── FORWARD PASS (what the model SEES) ───────────────────────────┐
│ Instruction: Analyze ... Input: tweet Output: positive <EOS>     │
│                                                                   │
│ ✓ All tokens flow through self-attention                         │
│ ✓ The hidden state of "positive" is computed from ALL previous   │
│   tokens, including the instruction and the tweet                │
│ ✓ The model builds full context: "after Output:, a label follows"│
└───────────────────────────────────────────────────────────────────┘

┌─── BACKWARD PASS (what the model LEARNS) ────────────────────────┐
│ Instruction: Analyze ... Input: tweet Output: positive <EOS>     │
│    -100      -100          -100   -100    [train]    [train]     │
│                                                                   │
│ ✗ No gradient from Instruction, Input, or Output: markers        │
│ ✓ Gradient ONLY from "positive" and <EOS>                        │
│ ✓ BUT: gradients flow THROUGH attention to update W_K, W_V, W_Q │
│   weights that processed the tweet (via chain rule — see below)  │
└───────────────────────────────────────────────────────────────────┘
```

**The model always reads the full instruction and tweet.** Masking only controls what the model is *graded on*.

---

#### The Chain Rule: Why Masked Tokens Still Get Updated

This is where most people get confused. They think: "If the tweet has `label = -100`, the model ignores the tweet entirely." That is **mathematically false**.

Let's walk through the calculus. Assume:

- Position $j$ = a tweet token (e.g., "love")
- Position $T$ = the sentiment label token (e.g., "positive")
- Only position $T$ contributes to the loss $L$

**Step 1: The loss depends only on the label position**

$$L = \text{CrossEntropy}(\text{logits}_T, \text{label}_T)$$

The gradient $\frac{\partial L}{\partial O_T} = \delta_T \neq 0$ exists only at position $T$.

**Step 2: The attention output at position $T$ is a weighted sum of ALL previous Value vectors**

$$O_T = \sum_{i=1}^{T} \alpha_i V_i$$

Where $\alpha_i = \text{softmax}\left(\frac{Q_T \cdot K_i^T}{\sqrt{d_k}}\right)_i$ is the attention weight, and $V_i = W_V \cdot h_i$ is the Value vector of token $i$.

Because the tweet at position $j$ is one of those indices, $O_T$ **depends on** $V_j$:

$$\frac{\partial O_T}{\partial V_j} = \alpha_j \neq 0$$

**Step 3: Gradient flows from the loss to the tweet's Value vector**

Using the chain rule:

$$\frac{\partial L}{\partial V_j} = \frac{\partial L}{\partial O_T} \cdot \frac{\partial O_T}{\partial V_j} = \delta_T \cdot \alpha_j \neq 0$$

This is non-zero! Even though position $j$ itself has no direct loss.

**Step 4: Gradient flows to the weight matrix $W_V$**

Since $V_j = W_V \cdot h_j$:

$$\frac{\partial L}{\partial W_V} = \sum_{i=1}^{T} \frac{\partial L}{\partial V_i} \cdot \frac{\partial V_i}{\partial W_V}$$

The tweet token contributes $\frac{\partial L}{\partial V_j} \cdot h_j^T$ to this sum — a **non-zero** term.

**Step 5: Same logic applies to $W_K$ and $W_Q$**

The attention weight $\alpha_j$ depends on $K_j = W_K \cdot h_j$:

$$\frac{\partial L}{\partial W_K} = \frac{\partial L}{\partial \alpha_j} \cdot \frac{\partial \alpha_j}{\partial K_j} \cdot \frac{\partial K_j}{\partial W_K} \neq 0$$

Every weight matrix that processes the tweet ($W_V, W_K, W_Q$) receives non-zero gradients because the label token *attends to* the tweet.

---

#### Summary Table

| Part of Sequence | In `input_ids`? | In `labels`? | Contributes to Loss? | Receives Gradient? |
|---|---|---|---|---|
| `Instruction:` ... | ✓ Yes | `-100` | ✗ No | ✓ Yes (via attention chain rule) |
| `Input:` (tweet) | ✓ Yes | `-100` | ✗ No | ✓ **Yes** (via attention chain rule) |
| `\nOutput:` marker | ✓ Yes | `-100` | ✗ No | ✓ Yes (via attention chain rule) |
| Sentiment label | ✓ Yes | Real ID | ✓ Yes | ✓ Yes |
| `<EOS>` | ✓ Yes | Real ID | ✓ Yes | ✓ Yes |
| `[PAD]` tokens | ✓ Yes | `-100` | ✗ No | ✗ No (attention mask blocks them) |

**The critical row**: the tweet tokens do not generate their own loss, but the weight matrices that **read** them are updated through backpropagation. The model learns to *interpret* the tweet, not to *reproduce* it.

---

#### Why This is Standard Practice

Loss masking is the **industry norm** for instruction fine-tuning. Here's why:

| Approach | Pros | Cons |
|---|---|---|
| **No masking** (old code) | Simple to implement | Model learns to copy template; hallucinates boilerplate at inference; wastes capacity on unhelpful predictions |
| **Loss masking** (our fix) | Model focuses on the answer; cleaner outputs; standard for LLaMA-2-Chat, Mistral-Instruct, etc. | Slightly more code; requires finding the `\nOutput:` boundary |

---

#### Why Alignment is NOT Broken

Alignment means: **training prompt format ≡ inference prompt format**. Loss masking does not change the prompt format — the model still sees:

```
Instruction: Analyze the sentiment ...\nInput: @user tweet\nOutput:
```

at **both** training and inference time. The model's self-attention mechanism builds the same contextual representations in both cases.

What changes is what the model is *rewarded* for predicting. Without masking, it's rewarded for predicting `Input:` after the label. With masking, it's only rewarded for predicting the label. The *understanding* of the task is the same — only the *output behavior* is different.

**Analogy**: A student reads the full exam question (forward pass). The teacher only grades the final answer (loss calculation). The student still learns to read the question carefully because getting the right answer *depends* on understanding it — the grade on the answer flows back to the study habits (backpropagation through attention).

## 10. Use the standard Hugging Face training workflow

We set up the **data collator** — a utility that dynamically batches tokenized sequences together. `DataCollatorForLanguageModeling` handles:

- **Dynamic padding**: Instead of padding all 5,000 sequences to 512 tokens (wasteful), it pads only within each batch to the longest sequence in that batch.
- **Label preparation**: For causal LM (`mlm=False`), it ensures labels are properly aligned for next-token prediction.
- **Tensor conversion**: Converts lists of token IDs into PyTorch tensors ready for the model.

The cell below imports all remaining Hugging Face classes and creates the data collator.

In [ ]:
# =============================================================================
# SECTION 10: Set up the Hugging Face Trainer and data collator
# =============================================================================
# DataCollatorForLanguageModeling dynamically batches tokenized sequences,
# applies padding within each batch, and handles the label shifting for
# causal language modeling (next-token prediction).

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

# DataCollatorForLanguageModeling: pads sequences to the longest in the batch
# and prepares labels for causal LM training
# mlm=False: we are NOT doing masked language modeling (like BERT)
# — this is causal (autoregressive) LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # causal LM — predict next token, not masked token
)

## 11. Set the training arguments

`TrainingArguments` is the central configuration object that controls every aspect of the training loop. Here's what each key setting means:

| Setting | Value | Explanation |
|---|---|---|
| `output_dir` | `./results` | Directory for checkpoints and logs |
| `per_device_train_batch_size` | 4 | 4 samples per GPU per step |
| `gradient_accumulation_steps` | 4 | Accumulate gradients for 4 steps → effective batch = 16 |
| `learning_rate` | 2e-4 | Learning rate for LoRA (higher than full fine-tuning) |
| `fp16` | `True` (GPU) | 16-bit mixed precision — faster, less memory |
| `num_train_epochs` | 3 | Three full passes through the 5,000-sample dataset |
| `save_strategy` | `"epoch"` | Save a checkpoint after every epoch |
| `optim` | `paged_adamw_8bit` (GPU) | Memory-efficient 8-bit AdamW optimizer |
| `report_to` | `"none"` | No external logging (no wandb/tensorboard needed) |

The cell below creates the `TrainingArguments` object with these settings.

In [ ]:
# =============================================================================
# SECTION 11: Configure training hyperparameters
# =============================================================================
# TrainingArguments controls every aspect of the training loop:
# batch size, learning rate, precision, logging, checkpointing, etc.

import torch
from transformers import TrainingArguments

use_cuda = torch.cuda.is_available()

training_args = TrainingArguments(
    # ---- Output & logging ----
    output_dir="./results",               # checkpoints and logs saved here
    logging_steps=3,                      # log training loss every 3 steps
    # logging_steps=10,                   # (alternative: log less frequently)
    report_to="none",                     # don't send logs to wandb/tensorboard

    # ---- Batch size ----
    per_device_train_batch_size=4,        # 4 samples per GPU per forward pass
    gradient_accumulation_steps=4,        # accumulate gradients over 4 steps
                                          # → effective batch size = 4 × 4 = 16

    # ---- Optimization ----
    learning_rate=2e-4,                   # learning rate for AdamW optimizer
    optim="paged_adamw_8bit" if use_cuda else "adamw_torch",
                                          # paged_adamw_8bit: memory-efficient 8-bit Adam for GPU
                                          # adamw_torch: standard AdamW for CPU

    # ---- Precision ----
    fp16=use_cuda,                        # fp16 mixed precision on GPU (faster, less memory)
    bf16=False,                           # bfloat16 disabled (use fp16 instead)

    # ---- Training duration ----
    num_train_epochs=3,                   # 3 full passes through the 5,000-sample dataset
    save_strategy="epoch",                # save a checkpoint after each epoch
)

## 12. Start training

On a 4090-class GPU this takes about 16 minutes (an RTX 5090 laptop GPU is similar). Training on CPU alone is possible but extremely slow.

In [ ]:
# =============================================================================
# SECTION 12: Create the Trainer and start fine-tuning
# =============================================================================
# The Hugging Face Trainer handles the entire training loop:
# forward pass → loss computation → backward pass → optimizer step → logging.
# On a 4090-class GPU this takes ~16 minutes for 3 epochs on 5,000 samples.

# Create the Trainer with our model, training config, dataset, and data collator
trainer = Trainer(
    model=model,                        # LoRA-wrapped, 4-bit quantized model
    args=training_args,                 # hyperparameters from Section 11
    train_dataset=tokenized_dataset,    # tokenized + labeled dataset
    data_collator=data_collator,        # dynamic batching & label prep
)

# Start the training loop — this is where the actual fine-tuning happens
# The Trainer will output loss values at each logging step
trainer.train()

## 13. Save the fine-tuned model

After training completes, we save the **LoRA adapter weights** (NOT the full 1.5B model). The adapter is tiny — typically a few megabytes — because it only contains the low-rank matrices (`A` and `B`) that were added to the attention layers.

The saved directory `./models/DeepSeek1.5B_finetuned/` will contain:
- `adapter_config.json` — LoRA configuration (rank, alpha, target modules, etc.)
- `adapter_model.safetensors` — the trained LoRA weights

At inference time, you load the base model and then attach the adapter with `PeftModel.from_pretrained()`. This is much more disk-efficient than saving a full 3 GB model copy.

In [ ]:
# =============================================================================
# SECTION 13: Save the fine-tuned LoRA adapter
# =============================================================================
# Only the LoRA adapter weights are saved (not the full 1.5B base model).
# The adapter is tiny (~a few MB) and can be loaded on top of the base model
# at inference time using PeftModel.from_pretrained().

# Output directory for the LoRA adapter weights
output_dir = "./models/DeepSeek1.5B_finetuned"
# Save the LoRA adapter weights (adapter_config.json + adapter_model.safetensors)
model.save_pretrained(output_dir)
# Save the tokenizer alongside (in case we added special tokens)
tokenizer.save_pretrained(output_dir)
print(f"LoRA adapter saved to {output_dir}")

## 14. Run a demonstration with the fine-tuned model

Now we test the fine-tuned model end-to-end on a **real tweet from the dataset**. The cell below:

1. **Loads the base model** with the same 4-bit quantization used during training
2. **Attaches the LoRA adapter** via `PeftModel.from_pretrained()` — this applies our fine-tuned weights on top of the frozen base model
3. **Picks a real tweet** from `first_50` (the training data loaded in Section 6) so we have a known ground-truth label to compare against
4. **Formats the prompt** in the exact same instruction template used during training: `Instruction: ...\nInput: <tweet>\nOutput:`
5. **Generates a prediction** — the model should output a sentiment label (positive/negative/neutral)
6. **Compares predicted vs actual** — prints both labels and whether they match

The key insight: the prompt format at inference must **exactly match** the format used during training (Section 7). If you change the template, the model may produce unexpected output because it was never trained on that format.

Also note the low temperature (`temperature=0.1`) — sentiment classification is a deterministic task, so we want the most confident answer, not creative variation.

In [ ]:
# =============================================================================
# SECTION 14: Run a sentiment analysis demo with the fine-tuned model
# =============================================================================
# This cell demonstrates the end-to-end inference pipeline:
# 1. Load the base model (with same quantization as training)
# 2. Attach the LoRA adapter (the fine-tuned weights)
# 3. Format a tweet in the same instruction template
# 4. Generate a sentiment prediction

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel                        # ← loads LoRA adapter onto base model

# ---- Paths ----
# Base model: the original DeepSeek 1.5B weights (downloaded via HF mirror)
base_model_name = r"D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B"
# Adapter: the LoRA weights we just fine-tuned and saved
adapter_path = "./models/DeepSeek1.5B_finetuned"

use_cuda = torch.cuda.is_available()

# ---- 4-bit quantization (must match the training configuration) ----
# This is the same BitsAndBytesConfig used during training in Section 8
# Optional: 4-bit quantization (same as during training) — GPU only
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# ---- Load tokenizer from the base model ----
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
# Set pad_token to eos_token — required for batched generation
tokenizer.pad_token = tokenizer.eos_token   # important for generation

# ---- Load base model (with quantization on GPU, without on CPU) ----
if use_cuda:
    # GPU: load with 4-bit quantization (matches training setup)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,    # remove if you didn't use quantization
        device_map="auto",
        trust_remote_code=True
    )
else:
    # CPU: cannot use BitsAndBytes; fall back to bfloat16
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

# ---- Attach the LoRA adapter to the base model ----
# PeftModel.from_pretrained loads the adapter weights and applies them on top
# of the frozen base model. The result behaves like a fine-tuned model.
model = PeftModel.from_pretrained(base_model, adapter_path)

# Switch to evaluation mode — disables dropout, etc.
model.eval()


In [ ]:

# ---- Test the model with a RANDOM tweet from the dataset ----
# We randomly pick a tweet from the training data (first_50 DataFrame, loaded in Section 6)
# so we can compare the model's prediction against the known ground-truth label.
# Using .sample() ensures the result isn't biased by position in the dataset.
sample_row = first_50.sample(n=1, random_state=None).iloc[0]  # random selection
tweet = sample_row['content']
actual_sentiment = sample_row['sentiment']

instruction = "Analyze the sentiment of the following tweet:"

# Format the prompt EXACTLY as during training (Section 7), but END at "Output:"
# — the model will complete the sequence by generating the sentiment label.
prompt = f"Instruction: {instruction}\nInput: {tweet}\nOutput:"

# Tokenize the prompt and move tensors to the same device as the model
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate the model's response
# torch.no_grad() disables gradient computation (saves memory during inference)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,            # sentiment label is short — limit to 10 tokens
        temperature=0.1,              # low temperature → more deterministic output
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# ---- Decode only the NEWLY GENERATED part (skip the input prompt) ----
# outputs[0] contains both prompt tokens + generated tokens
# inputs.input_ids.shape[1] gives the length of the prompt in tokens
# We slice from that index onward to get only the generated text
generated_ids = outputs[0][inputs.input_ids.shape[1]:]
# Strip whitespace & newlines — the model may append "\n" or extra spaces
predicted_sentiment = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"Tweet:               {tweet}")
print(f"Actual sentiment:    {actual_sentiment}")
print(f"Predicted sentiment: {predicted_sentiment}")
print(f"Match: {actual_sentiment.lower() == predicted_sentiment.lower()}")

## 15. Compare the fine-tuned model with the original dataset

This section evaluates the fine-tuned model more rigorously: it **randomly selects 5 tweets** from the full dataset, runs inference on each, and compares the predicted sentiment against the **ground-truth label**.

The cell below:
1. Loads the fine-tuned model (same steps as Section 14)
2. Re-loads the CSV and randomly samples 5 tweets (using `random_state=42` for reproducibility)
3. For each tweet, formats the prompt, generates a prediction, and compares with the actual label
4. Prints a ✓ or ✗ for each tweet
5. Computes a simple accuracy score at the end

In a full evaluation workflow, you would:
- Run inference on a held-out test set (tweets the model never saw during training)
- Compute accuracy, precision, recall, and F1 score per class
- Analyze where the model makes mistakes (e.g., confusing neutral with negative)

In [ ]:
# =============================================================================
# SECTION 15: Compare the fine-tuned model against the original dataset
# =============================================================================
# This cell loads both the fine-tuned model and the original CSV dataset,
# then runs inference on 5 randomly selected tweets to compare predicted
# vs actual sentiment labels. This is a basic evaluation of fine-tuning quality.

import torch
import os
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# --------------------------
# 1. Paths and model loading
# --------------------------
# Base model: original DeepSeek 1.5B weights
base_model_name = r"D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B"   # downloaded via HF mirror
# Adapter: our fine-tuned LoRA weights
adapter_path = "./models/DeepSeek1.5B_finetuned"

use_cuda = torch.cuda.is_available()

# Use the same 4-bit quantization config as during training (GPU only)
# If you used 4-bit quantization during training, load with same config (GPU only)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer from base model
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
# Set pad_token to eos_token — required for generator compatibility
tokenizer.pad_token = tokenizer.eos_token   # important for generation

# Load base model with or without quantization depending on GPU availability
if use_cuda:
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,    # remove if you didn't use quantization
        device_map="auto",
        trust_remote_code=True
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

# Attach the fine-tuned LoRA adapter to the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()   # inference mode — disables dropout and other training-only behavior

# --------------------------
# 2. Randomly select tweets and compare predictions vs ground truth
# --------------------------

# Re-load the CSV dataset (or reuse first_50 if still in memory)
csv_path = "twitter-airline-sentimentSentiment_Analysis.csv"
if not os.path.exists(csv_path):
    csv_path = r"C:\Deepin\Programming\20260803 AgenticAILLMVisionModel2026Tutorials\tutorials\01-llm-transformer-training\twitter-airline-sentimentSentiment_Analysis.csv"

df = pd.read_csv(csv_path)

# Randomly select 5 tweets — random_state=42 makes this reproducible
sample_df = df.sample(n=5, random_state=42)

instruction = "Analyze the sentiment of the following tweet:"

print("=" * 70)
print("COMPARING PREDICTIONS vs GROUND TRUTH (5 random tweets)")
print("=" * 70)

correct = 0
total = 0

for idx, row in sample_df.iterrows():
    tweet = row['content']
    actual = row['sentiment']

    # Format prompt with the SAME training template (Section 7)
    prompt = f"Instruction: {instruction}\nInput: {tweet}\nOutput:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,              # sentiment label is very short
            temperature=0.1,                # low temperature → deterministic output
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract only the generated tokens and clean the output
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    predicted = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # Compare (case-insensitive — model may capitalize differently)
    is_correct = actual.lower() == predicted.lower()
    if is_correct:
        correct += 1
    total += 1

    # Truncate tweet for display if it's very long
    display_tweet = tweet[:80] + ('...' if len(tweet) > 80 else '')
    print(f"\nTweet: {display_tweet}")
    print(f"  Actual:    {actual}")
    print(f"  Predicted: {predicted}")
    print(f"  {'✓ CORRECT' if is_correct else '✗ WRONG'}")

print(f"\n{'=' * 70}")
print(f"Accuracy: {correct}/{total} = {correct/total:.1%}")

## 16. Finish up

1. The LoRA adapter is saved in `./models/DeepSeek1.5B_finetuned` (relative to this notebook folder).

2. The base model lives in `D:\HuggingfaceDownload\DeepSeek-R1-Distill-Qwen-1.5B` — keep it for reuse; there is no need to re-download it.

3. (Optional) Merge the LoRA adapter into the base model to produce a standalone model:

   ```python
   from peft import PeftModel
   merged = PeftModel.from_pretrained(base_model, adapter_path).merge_and_unload()
   merged.save_pretrained("./models/DeepSeek1.5B_finetuned_merged")
   ```

At this point, the example of fine-tuning a billion-parameter model on a single card is complete.

These fine-tuned models can be deployed efficiently locally, greatly reducing both API latency and cost. The same method can be directly extended to tens-of-billions-scale models within 80 GB, such as 7B LLMs.

### Contact

For jobs or project collaboration: `yucongcai_business@outlook.com`  
For research-related inquiries: `yucongcai_research@outlook.com`

---

## Version log

| Version | Date | Change |
|---|---|---|
| v1.0 | 2026-08-03 | Original record of the LoRA fine-tuning notebook (kept in `assets/` as a reference copy). |